In [1]:
import os
import numpy as np

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import ollama

In [2]:
pdf_path = r"C:\Users\srish\Downloads\sample_academic_regulations.pdf"

reader = PdfReader(pdf_path)

text = ""

for page in reader.pages:
    page_text = page.extract_text()
    
    if page_text:
        text += page_text + "\n"

print("PDF loaded successfully!")
print("Characters:", len(text))

PDF loaded successfully!
Characters: 5242


In [5]:
def create_chunks(text, chunk_size=1000, overlap=200):
    chunks = []
    
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    
    return chunks

In [7]:
chunks = create_chunks(text)

print("Number of chunks:", len(chunks))

Number of chunks: 7


In [9]:
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded!


In [11]:
embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True,
    show_progress_bar=False
)

print("Embeddings created!")
print("Shape:", embeddings.shape)

Embeddings created!
Shape: (7, 384)


In [13]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (
        np.linalg.norm(a) * np.linalg.norm(b)
    )

In [15]:
def retrieve_context(question, top_k=3):
    
    question_embedding = embedding_model.encode(
        question,
        convert_to_numpy=True
    )
    
    scores = []
    
    for i, chunk_embedding in enumerate(embeddings):
        
        similarity = cosine_similarity(
            question_embedding,
            chunk_embedding
        )
        
        scores.append((similarity, i))
    
    scores.sort(reverse=True)
    
    results = []
    
    for score, index in scores[:top_k]:
        results.append({
            "score": score,
            "text": chunks[index]
        })
    
    return results

In [19]:
results = retrieve_context(question)

for i, result in enumerate(results):
    
    print("=" * 60)
    print("Context", i + 1)
    print("Similarity:", round(result["score"], 4))
    print()
    print(result["text"])

Context 1
Similarity: 0.5986

and administrative obligations.
13. Frequently Asked Academic Facts
Question
Answer
Minimum attendance for examination
75%
Attendance range eligible for possible condonation
65%–74%
Attendance below which repetition may be required
Below 65%
Minimum combined marks to pass a theory course
50%
Minimum end-semester examination component
40%

Question
Answer
Duration of a normal theory examination
3 hours
 End of Sample Academic Regulations


Context 2
Similarity: 0.5426

SAMPLE UNIVERSITY
Academic Regulations and Examination Handbook
 For Undergraduate Programmes
This sample document is created for demonstrating a Retrieval-Augmented Generation (RAG) system. It
contains fictional academic regulations and should not be treated as an official university policy.
1. General Academic Regulations
Students enrolled in undergraduate programmes must complete all prescribed courses, laboratory
requirements, assessments, and examinations for the award of the degree. Eac